# factorlib demo — 四種因子類別 × 完整 API 展示

這份 notebook 不是教學文檔，是**可執行的功能索引**：每個 cell 對應 `factorlib/README.md` 的一段 API，照順序跑完就看過 factorlib 全部功能。

涵蓋：
- 四種 `factor_type`：`cross_sectional` / `event_signal` / `macro_panel` / `macro_common`
- 六個使用層級（Level 0–6）：單因子 → 自訂 config → 個別 metrics → batch + BHY → redundancy → charts → MLflow
- 資料：`tw_stock_daily_2017_2025.parquet`（TW 全市場日線），切出 2023-01–2024-06 當 demo 視窗。

## 0. Setup

In [1]:
from __future__ import annotations

import polars as pl
import factorlib as fl

pl.Config.set_tbl_rows(8)
pl.Config.set_fmt_str_lengths(60)
print('factorlib version:', getattr(fl, '__version__', 'dev'))

factorlib version: dev


In [2]:
raw = pl.read_parquet('../tw_stock_daily_2017_2025.parquet')
raw

ticker,date,open_adj,high_adj,low_adj,close_adj,volume,market,market_cap,twse_ind
str,date,f64,f64,f64,f64,i64,str,f64,str
"""1101""",2017-01-03,17.7678,17.7678,17.442,17.6174,2890982,"""TWSE""",129780.0,"""水泥工業"""
"""1102""",2017-01-03,16.1136,16.144,16.0222,16.144,1271792,"""TWSE""",89078.0,"""水泥工業"""
"""1103""",2017-01-03,5.9821,5.9821,5.9134,5.9821,278853,"""TWSE""",6748.0,"""水泥工業"""
"""1104""",2017-01-03,14.1768,14.2636,14.1479,14.2347,150691,"""TWSE""",15610.0,"""水泥工業"""
…,…,…,…,…,…,…,…,…,…
"""9955""",2025-12-31,31.15,31.35,30.5,30.5,602215,"""TWSE""",3185.0,"""綠能環保"""
"""9958""",2025-12-31,140.0,140.0,137.0,139.5,592032,"""TWSE""",34442.0,"""鋼鐵工業"""
"""9960""",2025-12-31,21.2,21.45,21.05,21.45,52433,"""OTC""",721.0,"""運動休閒"""
"""9962""",2025-12-31,10.2,10.35,10.2,10.35,170162,"""OTC""",934.0,"""鋼鐵工業"""


In [3]:
# 載入原始 TW 日線，用 fl.adapt 把 user 欄位名對應到 factorlib canonical names
# (canonical = date / asset_id / price)
raw = pl.read_parquet('../tw_stock_daily_2017_2025.parquet')
raw_demo = fl.adapt(raw, date='date', asset_id='ticker', price='close_adj')

# 切出 2023-01 ~ 2024-06 demo 視窗（控制 runtime）
raw_demo = raw_demo.filter(
    (pl.col('date') >= pl.datetime(2023, 1, 1))
    & (pl.col('date') < pl.datetime(2024, 7, 1))
)
print('rows:', raw_demo.height, '| assets:', raw_demo['asset_id'].n_unique())
raw_demo.head(3)

rows: 639352 | assets: 1837


asset_id,date,open_adj,high_adj,low_adj,price,volume,market,market_cap,twse_ind
str,datetime[ms],f64,f64,f64,f64,i64,str,f64,str
"""1101""",2023-01-03 00:00:00,30.9445,31.0827,30.6683,30.8524,11654098,"""TWSE""",239732.0,"""水泥工業"""
"""1102""",2023-01-03 00:00:00,34.9668,34.9668,34.4545,34.668,3878899,"""TWSE""",143965.0,"""水泥工業"""
"""1103""",2023-01-03 00:00:00,15.6454,15.6906,15.6002,15.6906,88117,"""TWSE""",13442.0,"""水泥工業"""


## 1. 四種因子類別總覽

`fl.describe_factor_types()` 印出所有註冊的 factor_type 與用途；
`fl.describe_profile(<type>)` 反射對應 Profile dataclass 的欄位、canonical p 與方法。
這兩個是「不用開檔就知道 factorlib 現在長什麼樣」的入口。

In [4]:
fl.describe_factor_types()
print()
print('list_factor_types ->', fl.list_factor_types())

  cross_sectional     : 截面因子（每期每資產有 signal，N ≥ 30）→ 選股
  event_signal        : 事件訊號（離散觸發）→ 事件交易
  macro_panel         : 宏觀 panel（小截面 N < 30）→ 跨國配置
  macro_common        : 宏觀共用（單一時序）→ 風險歸因

list_factor_types -> ['cross_sectional', 'event_signal', 'macro_panel', 'macro_common']


In [5]:
for ft in fl.list_factor_types():
    fl.describe_profile(ft)


  cross_sectional — CrossSectionalProfile
  ──────────────────────────────────────────────────
  Typed profile for a cross-sectional factor.

  Fields:
    factor_name                  str
    n_periods                    int
    ic_mean                      float
    ic_tstat                     float
    ic_p                         PValue
    ic_ir                        float
    hit_rate                     float
    hit_rate_p                   PValue
    ic_trend                     float
    ic_trend_p                   PValue
    monotonicity                 float
    q1_q5_spread                 float
    spread_tstat                 float
    spread_p                     PValue
    q1_concentration             float
    q1_concentration_eff_ratio   float
    oos_survival_ratio           float
    oos_sign_flipped             bool
    median_universe_n            int
    orthogonalize_r2_mean        float | None
    orthogonalize_n_base         int
    regime_ic_min_tstat   

## 2. Cross-sectional（選股因子）

訊號型態：每期每資產有連續值。典型用法 = 動量 / 價值 / 規模。
Canonical test: `ic_p`（IC 非重疊 t-test）。

### 2.1 Build factor（用 factorlib 內建 generator）

In [6]:
from factorlib.factors import generate_momentum, generate_momentum_60d
from factorlib.factors.volatility import generate_volatility

mom20_raw = generate_momentum(raw_demo, lookback=20)
mom20 = fl.preprocess(mom20_raw, config=fl.CrossSectionalConfig())
print('columns:', mom20.columns)
print('shape:', mom20.shape)
mom20.select(['asset_id', 'date', 'price', 'factor', 'forward_return']).head(3)


columns: ['asset_id', 'date', 'open_adj', 'high_adj', 'low_adj', 'price', 'volume', 'market', 'market_cap', 'twse_ind', 'factor']
shape: (602684, 11)


asset_id,date,price,factor
str,datetime[ms],f64,f64
"""1101""",2023-02-10 00:00:00,33.9377,0.100002
"""1101""",2023-02-13 00:00:00,34.0758,0.104478
"""1101""",2023-02-14 00:00:00,34.3061,0.081277


### 2.2 Level 0 — 最簡單 `fl.evaluate`

一行跑完 preprocess + all metrics + diagnostics，回傳 `CrossSectionalProfile`（frozen dataclass）。

In [7]:
profile = fl.evaluate(mom20, 'Mom_20D', factor_type='cross_sectional')

print('type:       ', type(profile).__name__)
print('verdict:    ', profile.verdict())
print('canonical_p:', f'{profile.canonical_p:.4f}')
print('ic_mean:    ', f'{profile.ic_mean:+.4f}')
print('ic_tstat:   ', f'{profile.ic_tstat:+.2f}')
print('ic_ir:      ', f'{profile.ic_ir:+.3f}')
print('q1_q5 spread:', f'{profile.q1_q5_spread:+.4f}')
print('turnover:   ', f'{profile.turnover:.2%}')
print('net_spread: ', f'{profile.net_spread:+.4f}')

type:        CrossSectionalProfile
verdict:     FAILED
canonical_p: 0.7251
ic_mean:     -0.0098
ic_tstat:    -0.35
ic_ir:       -0.103
q1_q5 spread: +0.0009
turnover:    7.34%
net_spread:  +0.0005


In [8]:
# profile.diagnose() 回傳結構化 Diagnostic — 包含 severity / code / message
for d in profile.diagnose():
    print(f'[{d.severity:<7s}] {d.code:<35s} {d.message}')

[warn   ] cs.ic_weak_spread_strong            IC not significant (p > 0.05) but Q1-Q5 spread is — alpha may be driven by outlier stocks rather than the broad cross-section.
[veto   ] cs.oos_sign_flipped                 OOS sample shows a sign flip vs IS — overfitting risk; the factor did not generalize.


### 2.3 Level 1 — 自訂 `CrossSectionalConfig`

切換 forward horizon、quantile groups、trading-cost 估計等。

In [9]:
cfg = fl.CrossSectionalConfig(
    forward_periods=10,            # 10 日 forward return
    n_groups=5,                    # 5 組 quantile（Q1 自動取前 1/5）
    mad_n=3.0,
    return_clip_pct=(0.01, 0.99),
    estimated_cost_bps=30,
)
mom20_h10 = fl.preprocess(mom20_raw, config=cfg)
profile_10d = fl.evaluate(mom20_h10, 'Mom_20D_h10', config=cfg)
print('verdict @10d:', profile_10d.verdict(), '| ic_ir:', f'{profile_10d.ic_ir:+.3f}')


verdict @10d: FAILED | ic_ir: -0.044


### 2.4 Research session — `fl.factor()`

研究場景的主入口。`fl.factor(df, name, config=cfg)` 回一個 `Factor`
session object，把 df / name / config / artifacts cache 綁在一起，所有
standalone metric 都是 `f.<metric>()` 方法、**統一回 `MetricOutput`**。

| 入口 | 用途 |
|---|---|
| `fl.evaluate(df, name, cfg)` | 一步吐 Profile（production / batch / CI） |
| `fl.factor(df, name, cfg)` | 保留 handle、逐 metric 呼叫（research / drill-down / 敏感度） |

兩者等價——`fl.evaluate(df, name, cfg)` ≡ `fl.factor(df, name, cfg).evaluate()`，
且 cache 共用：先呼叫 `f.ic()` 再 `f.evaluate()` 不會重算。


In [ ]:
# bind once, call many — 14 個 metric 都暴露為方法、shape 一致
f = fl.factor(mom20_h10, 'Mom_20D', config=cfg)

for name in ['ic', 'ic_ir', 'hit_rate', 'quantile_spread', 'monotonicity', 'oos_decay']:
    m = getattr(f, name)()
    p = m.metadata.get('p_value')
    p_str = 'n/a' if p is None else f'{p:.4f}'
    stat_str = 'n/a' if m.stat is None else f'{m.stat:+.2f}'
    print(f'{name:<17s} value={m.value:+.4f}  stat={stat_str:>6s}  p={p_str}  sig={m.significance or ""}')


#### Per-call override — 不污染 cache

想對單一 metric 做參數敏感度？帶 kwarg 直接蓋 config 預設值。override
path **繞過 cache、結果不寫回**——下次 default 呼叫還是拿到
config-bound 結果，`f.evaluate()` 也不會被污染。


In [ ]:
default_spread = f.quantile_spread()              # config.n_groups
override_spread = f.quantile_spread(n_groups=3)   # per-call override

print(f'default  n_groups={cfg.n_groups}: value={default_spread.value:+.4f}')
print(f'override n_groups=3: value={override_spread.value:+.4f}')

# cache 還是 default 的 config-bound 結果
cached = f.artifacts.metric_outputs['q1_q5_spread']
print(f'cache untouched by override: {cached.value == default_spread.value}')


#### Cache share — `f.evaluate()` 接力已算過的 metric

研究常是「逐 metric 看完、再要完整 Profile」。`f.evaluate()` 會 reuse
所有已呼叫的 metric，**不重算**。反之亦然：`f.evaluate()` 跑完後
`f.ic()` 也是 cache hit。Profile 跟 standalone metric 共用單一
source-of-truth (`artifacts.metric_outputs`)。


In [ ]:
ic_before = f.artifacts.metric_outputs['ic']  # 在 cell 17 就算過了
profile = f.evaluate()                         # reuse，不重算
ic_after = f.artifacts.metric_outputs['ic']
print(f'ic identity preserved across evaluate(): {ic_before is ic_after}')
print(f'profile.ic_p: {profile.ic_p:.4f}')
print(f'cache size after evaluate: {len(f.artifacts.metric_outputs)} metrics cached')


#### Escape hatch — `f.artifacts` 接下游工具 + L2 short-circuit

`f.artifacts` 跟 `fl.evaluate(..., return_artifacts=True)` 拿到同一種
`Artifacts`。所有吃 artifacts 的工具（`describe_profile_values` /
`redundancy_matrix` / user-defined custom metric）直接接入。

Level-2 opt-in（`regime_ic` / `multi_horizon_ic` / `spanning_alpha`）沒配
config 時**不 raise，回 short-circuit MetricOutput**——uniform contract，
使用者不用到處寫 `if / None` guard。


In [ ]:
# describe_profile_values 走 Factor 的 artifacts
fl.describe_profile_values(profile, f.artifacts, include_detail=False)

# L2 opt-in 沒傳 regime_labels → short-circuit
regime = f.regime_ic()
print(f'\nregime_ic (unconfigured): value={regime.value}  '
      f'reason={regime.metadata["reason"]}')


#### Low-level primitive path — 寫客製 metric / 特殊研究情境用

`factorlib.metrics.*` 是底層，函式各自吃 prepared panel 或 processed
intermediate（`ic_series` / `caar_series`）。日常 research 走
`fl.factor(...)`；寫 custom metric / 做 library-internal 測試才下到這層。


In [ ]:
from factorlib.metrics import compute_ic, ic as ic_fn, quantile_spread

prepared = fl.preprocess(mom20_raw, config=cfg)
ic_series = compute_ic(prepared)
ic_raw = ic_fn(ic_series, forward_periods=cfg.forward_periods)
spread_raw = quantile_spread(prepared, n_groups=cfg.n_groups)

print('low-level ic    :', ic_raw)
print('low-level spread:', spread_raw)


### 2.4.1 Orthogonalization — 接回 Step 6 pipeline

傳 basis DataFrame（shortcut）或 `OrthoConfig`（進階）到 `ortho=`，pipeline 會對每個 date 做 residualization，並在 Profile 上 surface R²。`OrthoConfig()` 沒傳 `base_factors` 會直接 TypeError——防 Phase 1 false-provenance bug。


In [11]:
# Build a simple basis: log market cap proxy per (date, asset_id).
import numpy as np

basis_df = (
    raw_demo
    .filter((pl.col('price') > 0) & (pl.col('volume') > 0))
    .with_columns((pl.col('price') * pl.col('volume')).log().alias('size'))
    .select('date', 'asset_id', 'size')
)
print('basis rows:', basis_df.height, 'cols:', basis_df.columns)


basis rows: 635610 cols: ['date', 'asset_id', 'size']


In [12]:
# DataFrame shortcut — 最常見的 case，一個 kwarg 搞定。
cfg_ortho = fl.CrossSectionalConfig(forward_periods=5, ortho=basis_df)
mom20_ortho = fl.preprocess(mom20_raw, config=cfg_ortho)
p_ortho = fl.evaluate(mom20_ortho, 'Mom20_orth', config=cfg_ortho)

print(f'R² explained by basis: {p_ortho.orthogonalize_r2_mean:.3f}')
print(f'n_base factors      : {p_ortho.orthogonalize_n_base}')
print(f'ic_mean (raw)       : {profile.ic_mean:.4f}')
print(f'ic_mean (ortho)     : {p_ortho.ic_mean:.4f}  — residual 去掉 size echo')


R² explained by basis: 0.078
n_base factors      : 1
ic_mean (raw)       : -0.0098
ic_mean (ortho)     : -0.0053  — residual 去掉 size echo


In [13]:
# Advanced — explicit OrthoConfig 控制 cols / min_coverage.
cfg_ortho_adv = fl.CrossSectionalConfig(
    forward_periods=5,
    ortho=fl.OrthoConfig(
        base_factors=basis_df,
        cols=['size'],              # None = 用所有 non-key cols
        min_coverage=0.90,          # 預設 0.95 — 低於此 raise
    ),
)
p_ortho_adv = fl.evaluate(
    fl.preprocess(mom20_raw, config=cfg_ortho_adv), 'Mom20_orth_adv', config=cfg_ortho_adv,
)
print(f'ic_p (advanced) : {p_ortho_adv.ic_p:.4f}')


ic_p (advanced) : 0.9513


### 2.4.2 Level 2 metrics — 接入 Profile（T3.S2）

在 config 上傳 `regime_labels` / `multi_horizon_periods` / `spanning_base_spreads`，pipeline 會多算這些 metric 並在 Profile 上 surface 6 個 scalar 欄位。每個欄位獨立 opt-in，不傳就維持 `None`。


In [14]:
# Regime labels: 用全市場當日平均 return（從 raw_demo 收盤算）切 bull / bear。
daily_mean_ret = (
    raw_demo
    .sort(['asset_id', 'date'])
    .with_columns(pl.col('price').pct_change().over('asset_id').alias('ret'))
    .group_by('date').agg(pl.col('ret').mean().alias('mkt_ret'))
    .drop_nulls().sort('date')
)
median_ret = daily_mean_ret['mkt_ret'].median()
regime_df = (
    daily_mean_ret
    .with_columns(
        pl.when(pl.col('mkt_ret') >= median_ret)
        .then(pl.lit('bull'))
        .otherwise(pl.lit('bear'))
        .alias('regime')
    )
    .select('date', 'regime')
)
print(regime_df.head(3))


shape: (3, 2)
┌─────────────────────┬────────┐
│ date                ┆ regime │
│ ---                 ┆ ---    │
│ datetime[ms]        ┆ str    │
╞═════════════════════╪════════╡
│ 2023-01-04 00:00:00 ┆ bull   │
│ 2023-01-05 00:00:00 ┆ bear   │
│ 2023-01-06 00:00:00 ┆ bull   │
└─────────────────────┴────────┘


In [15]:
# 一次傳三個 Level 2 inputs（flat 在 CrossSectionalConfig 上）。
cfg_l2 = fl.CrossSectionalConfig(
    forward_periods=5,
    regime_labels=regime_df,
    multi_horizon_periods=[1, 5, 10, 20],
    # spanning_base_spreads 需要先跑另一個因子拿到 spread_series；下一 cell 展示
)
mom20_l2 = fl.preprocess(mom20_raw, config=cfg_l2)
p_l2 = fl.evaluate(mom20_l2, 'Mom20_l2', config=cfg_l2)

print(f'regime_ic_min_tstat        : {p_l2.regime_ic_min_tstat}')
print(f'regime_ic_consistent       : {p_l2.regime_ic_consistent}')
print(f'multi_horizon_ic_retention : {p_l2.multi_horizon_ic_retention}')
print(f'multi_horizon_ic_monotonic : {p_l2.multi_horizon_ic_monotonic}')


regime_ic_min_tstat        : -0.9574202209973406
regime_ic_consistent       : True
multi_horizon_ic_retention : -0.3498961055410109
multi_horizon_ic_monotonic : True


In [16]:
# Spanning: 測 mom20 相對 mom60 的 Q1-Q5 spread 是否還有 alpha.
mom60_local_raw = generate_momentum_60d(raw_demo)
mom60_local = fl.preprocess(mom60_local_raw, config=fl.CrossSectionalConfig())
p_mom60, arts_mom60 = fl.evaluate(
    mom60_local, 'Mom_60D',
    factor_type='cross_sectional',
    return_artifacts=True,
)
mom60_spread = arts_mom60.intermediates['spread_series'].select('date', 'spread')

cfg_span = fl.CrossSectionalConfig(
    forward_periods=5,
    spanning_base_spreads={'mom_60d': mom60_spread},
)
# return_artifacts=True 讓 spanning_alpha 的 per-base-factor betas / R² /
# n_obs 也能從 metric_outputs 拿到（Profile scalar 只有 t / p 兩欄）。
p_span, arts_span = fl.evaluate(
    fl.preprocess(mom20_raw, config=cfg_span), 'Mom20_vs_Mom60',
    config=cfg_span, return_artifacts=True,
)
print(f'spanning_alpha_t : {p_span.spanning_alpha_t}')
print(f'spanning_alpha_p : {p_span.spanning_alpha_p}')

# Raw MetricOutput drill-down — Profile 沒有的 per-base-factor betas 在這裡
span_md = arts_span.metric_outputs['spanning_alpha'].metadata
print(f"\nbetas     : {span_md['betas']}")
print(f"R²        : {span_md['r_squared']:.4f}")
print(f"n_obs     : {span_md['n_obs']}")


spanning_alpha_t : 0.2482048254876567
spanning_alpha_p : 0.8048679706463877

betas     : {'mom_60d': 0.7589974150365089}
R²        : 0.4633
n_obs     : 58


### 2.4.3 Drill-down views — `describe_profile_values`

Profile 欄位是 scalar summary（rank / filter / BHY 用）；per-regime /
per-horizon / spanning 的逐欄 detail 存在 `artifacts.metric_outputs`。
`fl.describe_profile_values(profile, arts)` 一次印 scalar + detail sections，
也可以用 `include_detail=False` 只看 scalar。

Raw 存取：`arts.metric_outputs["regime_ic"].metadata["per_regime"]` 這條路是
public、結構化（dict of dict），是目標驅動 drill-down（畫圖、寫報告）的入口。

Batch 用法：`arts_map[profile.factor_name]` 把 profile 對到它的 artifacts——
下一個 cell 示範。

In [17]:
# 重跑 cfg_l2 但拿 artifacts，給 describe_profile_values 用
p_l2_full, arts_l2 = fl.evaluate(
    mom20_l2, 'Mom20_l2', config=cfg_l2, return_artifacts=True,
)

# 單 factor 單行印 scalar + regime / multi-horizon detail
fl.describe_profile_values(p_l2_full, arts_l2)

# Raw MetricOutput 直接讀——summary 用 repr，per-bucket 細節用 metadata
regime_ic = arts_l2.metric_outputs['regime_ic']
print('\n', repr(regime_ic))
print(regime_ic.metadata['per_regime'])

multi_horizon = arts_l2.metric_outputs['multi_horizon_ic']
print('\n', repr(multi_horizon))
print(multi_horizon.metadata['per_horizon'])



  Mom20_l2 — CrossSectionalProfile  (n_periods=330, verdict=FAILED)
  ------------------------------------------------------------
  Metrics:
    ic                 value=-0.0098    stat=-0.3532  sig=     p=0.7251
    ic_ir              value=-0.1025    stat=-        sig=     p=-
    hit_rate           value=0.5000     stat=0.0000   sig=     p=1.0000
    ic_trend           value=-0.0001    stat=-1.3013  sig=     p=0.1941
    monotonicity       value=0.4645     stat=2.7904   sig=***  p=0.0069
    q1_q5_spread       value=0.0009     stat=2.0571   sig=**   p=0.0437
    turnover           value=0.0734     stat=-        sig=     p=-
    breakeven_cost     value=61.9188    stat=-        sig=     p=-
    net_spread         value=0.0005     stat=-        sig=     p=-
    q1_concentration   value=177.3400   stat=311.7139 sig=     p=1.0000

  Detail:
    regime_ic  [direction_consistent=True]
      regime    mean_ic     stat        p  sig  n
      bear      -0.0075  -0.9574   0.3398       166
 

In [18]:
# Batch: evaluate_batch(keep_artifacts=True) 回 (ProfileSet, arts_map)。
# arts_map key 用 factor_name → drill-down top-K 時用 arts_map[p.factor_name].
factors_map_l2 = {
    'Mom_20D': mom20_l2,
    'Mom_60_5': fl.preprocess(mom60_local_raw, config=cfg_l2),
}
ps_l2, arts_map_l2 = fl.evaluate_batch(
    factors_map_l2, config=cfg_l2, keep_artifacts=True,
)
for p in ps_l2.rank_by('ic_ir', descending=True).top(2):
    fl.describe_profile_values(
        p, arts_map_l2[p.factor_name], include_detail=False,
    )



  Mom_60_5 — CrossSectionalProfile  (n_periods=290, verdict=FAILED)
  ------------------------------------------------------------
  Metrics:
    ic                 value=0.0115     stat=0.8448   sig=     p=0.4018
    ic_ir              value=0.1242     stat=-        sig=     p=-
    hit_rate           value=0.5517     stat=0.7878   sig=     p=0.4308
    ic_trend           value=-0.0000    stat=-0.6509  sig=     p=0.5156
    monotonicity       value=0.4683     stat=2.8854   sig=***  p=0.0055
    q1_q5_spread       value=0.0008     stat=1.8802   sig=*    p=0.0652
    turnover           value=0.0257     stat=-        sig=     p=-
    breakeven_cost     value=148.0369   stat=-        sig=     p=-
    net_spread         value=0.0006     stat=-        sig=     p=-
    q1_concentration   value=177.9878   stat=558.0650 sig=     p=1.0000


  Mom_20D — CrossSectionalProfile  (n_periods=330, verdict=FAILED)
  ------------------------------------------------------------
  Metrics:
    ic        

### 2.5 Level 3 — `evaluate_batch` + BHY + rank + top

批次評估多因子，回傳 polars-native `ProfileSet`。
`multiple_testing_correct` 做 BHY 多重檢定校正；`filter` / `rank_by` / `top` 可鏈式串接。

In [19]:
# 準備三個候選 CS 因子
mom60_raw = generate_momentum_60d(raw_demo)
vol20_raw = generate_volatility(raw_demo, lookback=20)
mom60 = fl.preprocess(mom60_raw, config=fl.CrossSectionalConfig())
vol20 = fl.preprocess(vol20_raw, config=fl.CrossSectionalConfig())

factors_map = {
    'Mom_20D': mom20,
    'Mom_60_5': mom60,
    'Vol_20D': vol20,
}
ps = fl.evaluate_batch(factors_map, factor_type='cross_sectional')
print('ProfileSet size:', len(ps), '| profile class:', ps.profile_cls.__name__)
ps.to_polars().select(['factor_name', 'ic_mean', 'ic_ir', 'q1_q5_spread', 'canonical_p'])  # quick glance


ProfileSet size: 3 | profile class: CrossSectionalProfile


factor_name,ic_mean,ic_ir,q1_q5_spread,canonical_p
str,f64,f64,f64,f64
"""Mom_20D""",-0.009785,-0.102516,0.00091,0.725113
"""Mom_60_5""",0.011486,0.124206,0.000762,0.401779
"""Vol_20D""",0.048925,0.310812,-0.000532,0.01565


In [20]:
# BHY 多重檢定 + 按 IC_IR 排序 + 取 top 2
top = (
    ps
    .multiple_testing_correct(p_source='canonical_p', fdr=0.10)
    .rank_by('ic_ir', descending=True)
    .top(2)
)
top.to_polars().select([
    'factor_name', 'ic_p', 'p_adjusted', 'bhy_significant', 'ic_ir',
])

factor_name,ic_p,p_adjusted,bhy_significant,ic_ir
str,f64,f64,bool,f64
"""Vol_20D""",0.01565,0.086077,true,0.310812
"""Mom_60_5""",0.401779,1.0,false,0.124206


In [21]:
# 也能傳 pl.Expr 或 Callable[[Profile], bool] 給 filter
stable = ps.filter(pl.col('ic_ir').abs() > 0.1)
print('factors with |IC_IR| > 0.1:', [p.factor_name for p in stable])

factors with |IC_IR| > 0.1: ['Mom_20D', 'Mom_60_5', 'Vol_20D']


### 2.5.1 `with_extra_columns` — 自訂欄位接回 ProfileSet

算完後想把自己的 metric（earnings momentum、turnover adjusted Sharpe、custom rank）掛回 ProfileSet 一起 filter / rank。位置是 positional，所以記得用 `profiles.to_polars()['factor_name']` 的順序對齊。


In [22]:
import numpy as np

# 假設這是使用者外部算的 metric，keyed by factor_name.
external_scores = {'Mom_20D': 0.8, 'Mom_60_5': 0.4, 'Vol_20D': 0.6}
aligned = [external_scores[name] for name in ps.to_polars()['factor_name'].to_list()]

scored = ps.with_extra_columns({'analyst_score': aligned})
best = (
    scored
    .filter(pl.col('analyst_score') > 0.5)
    .rank_by('analyst_score')
)
print(best.to_polars().select('factor_name', 'ic_ir', 'analyst_score'))


shape: (2, 3)
┌─────────────┬───────────┬───────────────┐
│ factor_name ┆ ic_ir     ┆ analyst_score │
│ ---         ┆ ---       ┆ ---           │
│ str         ┆ f64       ┆ f64           │
╞═════════════╪═══════════╪═══════════════╡
│ Mom_20D     ┆ -0.102516 ┆ 0.8           │
│ Vol_20D     ┆ 0.310812  ┆ 0.6           │
└─────────────┴───────────┴───────────────┘


### 2.5.2 Large-batch two-stage screening（Level 3b）

候選因子一多（>100）就不該每個都跑完整 pipeline。兩階段：
1. **Stage 1** — cheap per-factor IC screen，收集 p-values，取 top-K
2. **Stage 2** — 對 survivors 跑 `evaluate_batch`，BHY 校正時傳 `n_total=` 讓分母反映**原始**候選數（不是 survivors），維持 FDR 控制

> **Caveat**：如果預篩 metric 跟 canonical_p 同家族（e.g. 用 ic_p 篩、對 ic_p 校正），BHY 控制的是 marginal FDR 不是 conditional FDR——回報的 FDR 是 conditional 的下界。研究 paper 要揭露。


In [23]:
from factorlib.metrics.ic import compute_ic, ic as ic_metric
from factorlib.factors import generate_momentum, generate_volatility

# 組 10 個 candidate 因子（lookback sweep）模擬 large-batch 場景
candidates = {}
for lb in [5, 10, 20, 40, 60]:
    candidates[f'mom_{lb}']  = generate_momentum(raw_demo, lookback=lb)
    candidates[f'vol_{lb}']  = generate_volatility(raw_demo, lookback=lb)
print(f'N candidates: {len(candidates)}')


N candidates: 10


In [24]:
# Stage 1: cheap IC screen (preprocess + compute_ic + ic metric, 跳過 full pipeline)
cfg_base = fl.CrossSectionalConfig(forward_periods=5)
stage1_rows = []
for name, df in candidates.items():
    prep = fl.preprocess(df, config=cfg_base)
    ic_m = ic_metric(compute_ic(prep), forward_periods=cfg_base.forward_periods)
    stage1_rows.append({'name': name, 'ic_p': float(ic_m.metadata['p_value'])})

stage1_df = pl.DataFrame(stage1_rows).sort('ic_p')
top_k = stage1_df.head(3)   # 取最顯著 top-3（真實場景會是 top-50 / top-100）
print('Stage 1 rankings:')
print(stage1_df)
print('\nTop-k survivors:', top_k['name'].to_list())


Stage 1 rankings:
shape: (10, 2)
┌────────┬──────────┐
│ name   ┆ ic_p     │
│ ---    ┆ ---      │
│ str    ┆ f64      │
╞════════╪══════════╡
│ mom_5  ┆ 0.000273 │
│ vol_60 ┆ 0.003814 │
│ vol_10 ┆ 0.010785 │
│ vol_40 ┆ 0.012727 │
│ …      ┆ …        │
│ mom_20 ┆ 0.725113 │
│ mom_10 ┆ 0.738133 │
│ mom_40 ┆ 0.78774  │
│ mom_60 ┆ 0.991411 │
└────────┴──────────┘

Top-k survivors: ['mom_5', 'vol_60', 'vol_10']


In [25]:
# Stage 2: full pipeline on survivors，BHY n_total 用 len(candidates)
survivors = fl.evaluate_batch(
    {n: fl.preprocess(candidates[n], config=cfg_base) for n in top_k['name'].to_list()},
    factor_type='cross_sectional',
)
adjusted = survivors.multiple_testing_correct(
    fdr=0.05, n_total=len(candidates),
)

print(adjusted.to_polars().select(
    'factor_name', 'ic_p', 'p_adjusted', 'bhy_significant', 'mt_n_total'
))


shape: (3, 5)
┌─────────────┬──────────┬────────────┬─────────────────┬────────────┐
│ factor_name ┆ ic_p     ┆ p_adjusted ┆ bhy_significant ┆ mt_n_total │
│ ---         ┆ ---      ┆ ---        ┆ ---             ┆ ---        │
│ str         ┆ f64      ┆ f64        ┆ bool            ┆ i32        │
╞═════════════╪══════════╪════════════╪═════════════════╪════════════╡
│ mom_5       ┆ 0.000273 ┆ 0.007984   ┆ true            ┆ 10         │
│ vol_60      ┆ 0.003814 ┆ 0.055853   ┆ false           ┆ 10         │
│ vol_10      ┆ 0.010785 ┆ 0.105299   ┆ false           ┆ 10         │
└─────────────┴──────────┴────────────┴─────────────────┴────────────┘


In [26]:
# Repeat call raises — 防止 chain BHY on already-adjusted p-values.
try:
    adjusted.multiple_testing_correct(fdr=0.10, n_total=len(candidates))
except RuntimeError as e:
    print(f'refused: {e}')


refused: ProfileSet already has multiple-testing correction applied (mt_method='bhy', mt_n_total=10). Re-applying BHY to already-adjusted p-values is statistically meaningless (the first correction is the authoritative one). Build a fresh ProfileSet from the raw profiles if you need to redo the correction with different parameters.


### 2.6 Level 4 — Redundancy matrix

多因子之間的成對 `|ρ|`，抓出冗餘訊號。

In [27]:
# redundancy_matrix 需要 per-factor Artifacts；用 keep_artifacts=True 保留。
ps2, arts = fl.evaluate_batch(
    factors_map, factor_type='cross_sectional', keep_artifacts=True,
)

redund = fl.redundancy_matrix(ps2, method='value_series', artifacts=arts)
redund

/Users/cfh00884862/Desktop/dst/code/factor-analysis/factorlib/metrics/redundancy.py:114: UserWarning: redundancy_matrix: factors ['Mom_60_5'] have missing dates that other factors cover; correlation is computed on the intersection. If that shrinks the window unacceptably, drop the short-history factors from the ProfileSet.
  return _value_series_matrix(names, artifacts, profiles.profile_cls)


factor,Mom_20D,Mom_60_5,Vol_20D
str,f64,f64,f64
"""Mom_20D""",1.0,0.495109,0.118649
"""Mom_60_5""",0.495109,1.0,0.484736
"""Vol_20D""",0.118649,0.484736,1.0


In [28]:
# 跑完 redundancy 挑最不冗餘的因子：arts_map[name] 把 profile 對到它的 artifacts
# 然後 describe_profile_values 一行看 scalar + detail——典型「pruning 完要深入看」流程
top_factor = next(iter(ps2.rank_by('ic_ir', descending=True).top(1)))
fl.describe_profile_values(top_factor, arts[top_factor.factor_name])


  Vol_20D — CrossSectionalProfile  (n_periods=330, verdict=PASS)
  ------------------------------------------------------------
  Metrics:
    ic                 value=0.0489     stat=2.4822   sig=**   p=0.0157
    ic_ir              value=0.3108     stat=-        sig=     p=-
    hit_rate           value=0.6212     stat=1.9695   sig=**   p=0.0489
    ic_trend           value=0.0000     stat=0.2122   sig=     p=0.8321
    monotonicity       value=0.6988     stat=-1.5711  sig=     p=0.1210
    q1_q5_spread       value=-0.0005    stat=-0.9214  sig=     p=0.3602
    turnover           value=0.0126     stat=-        sig=     p=-
    breakeven_cost     value=-211.1177  stat=-        sig=     p=-
    net_spread         value=-0.0006    stat=-        sig=     p=-
    q1_concentration   value=176.7520   stat=1214.8813 sig=     p=1.0000



### 2.7 Level 5 — Charts (optional dep: `factorlib[charts]`)

`build_artifacts` 保留中間結果（IC series / spread series / quantile group returns…），
丟進 `report_charts` 產 plotly 圖。未安裝 `plotly` 會 raise ImportError — 此處 try/except 包起來。

In [29]:
from factorlib.evaluation.pipeline import build_artifacts

cfg_charts = fl.CrossSectionalConfig()
prepared = fl.preprocess(mom20_raw, config=cfg_charts)
artifacts = build_artifacts(prepared, cfg_charts)
artifacts.factor_name = 'Mom_20D'
print('artifacts.intermediates keys:', sorted(artifacts.intermediates.keys()))

try:
    from factorlib.charts import report_charts
    figs = report_charts(artifacts)
    print(f'produced {len(figs)} figures:', list(figs)[:5])
    # 在 notebook 直接顯示第一張
    next(iter(figs.values())).show()
except ImportError as e:
    print('charts skipped — install with `pip install factorlib[charts]`:', e)


artifacts.intermediates keys: ['ic_series', 'ic_values', 'spread_series']


produced 4 figures: ['cumulative_ic', 'ic_distribution', 'spread_ts', 'quantile_returns']


### 2.8 Level 6 — MLflow tracking (optional dep: `factorlib[mlflow]`)

`on_result` callback 在 `evaluate_batch` 的每個因子算完後觸發，
可用來 log profile 到 MLflow experiment。這裡只示範寫法，不實跑（避免副作用）。

In [30]:
# 不實跑 — 只展示 API 形狀
demo = '''
from factorlib.integrations.mlflow import FactorTracker

tracker = FactorTracker('Factor_Zoo')
ps = fl.evaluate_batch(
    factors_map,
    factor_type='cross_sectional',
    on_result=lambda name, p: tracker.log_profile(p, factor_type='cross_sectional'),
)
'''
print(demo)


from factorlib.integrations.mlflow import FactorTracker

tracker = FactorTracker('Factor_Zoo')
ps = fl.evaluate_batch(
    factors_map,
    factor_type='cross_sectional',
    on_result=lambda name, p: tracker.log_profile(p, factor_type='cross_sectional'),
)



## 3. Event-signal（事件交易因子）

訊號型態：離散觸發 `{-1, 0, +1}`。Canonical test: `caar_p`（CAAR 非重疊 t-test）。
範例：黃金交叉 / 死亡交叉 → 事件後 5 日有異常報酬嗎？

### 3.1 Build golden/death-cross event signal

In [31]:
event_df = (
    raw_demo.sort(['asset_id', 'date'])
    .with_columns(
        pl.col('price').rolling_mean(5).over('asset_id').alias('ma5'),
        pl.col('price').rolling_mean(20).over('asset_id').alias('ma20'),
    )
    .with_columns(
        pl.when(pl.col('ma5') > pl.col('ma20')).then(1)
          .otherwise(-1).alias('cross_state')
    )
    .with_columns(
        (pl.col('cross_state') - pl.col('cross_state').shift(1).over('asset_id')).alias('delta')
    )
    .with_columns(
        pl.when(pl.col('delta') == 2).then(1.0)   # 黃金交叉
        .when(pl.col('delta') == -2).then(-1.0)   # 死亡交叉
        .otherwise(0.0).alias('factor')
    )
    .filter(pl.col('factor').is_not_null() & pl.col('ma20').is_not_null())
    .select(['asset_id', 'date', 'price', 'factor'])
)
event_df_raw = event_df
print('factor value counts:')
print(event_df['factor'].value_counts().sort('factor'))

factor value counts:
shape: (3, 2)
┌────────┬────────┐
│ factor ┆ count  │
│ ---    ┆ ---    │
│ f64    ┆ u32    │
╞════════╪════════╡
│ -1.0   ┆ 19738  │
│ 0.0    ┆ 564016 │
│ 1.0    ┆ 20760  │
└────────┴────────┘


### 3.2 evaluate（自動切到 `EventProfile`）

In [32]:
ev_cfg = fl.EventConfig(
    forward_periods=5,
    event_window_pre=5,
    event_window_post=20,
    cluster_window=3,
    adjust_clustering='none',
)
event_df = fl.preprocess(event_df_raw, config=ev_cfg)
ev_profile = fl.evaluate(event_df, 'GoldenCross', config=ev_cfg)

print('type:           ', type(ev_profile).__name__)
print('n_events:       ', ev_profile.n_events)
print('n_periods (dates):', ev_profile.n_periods)
print('verdict:        ', ev_profile.verdict())
print('CAAR mean:      ', f'{ev_profile.caar_mean:+.4f}')
print('CAAR p (canonical):', f'{ev_profile.caar_p:.4f}')
print('BMP z:          ', f'{ev_profile.bmp_zstat:+.2f}  p={ev_profile.bmp_p:.4f}')
print('event_hit_rate: ', f'{ev_profile.event_hit_rate:.2%}  p={ev_profile.event_hit_rate_p:.4f}')
print('profit_factor:  ', f'{ev_profile.profit_factor:.3f}')
print('clustering_hhi (normalized):', ev_profile.clustering_hhi_normalized)


type:            EventProfile
n_events:        39775
n_periods (dates): 331
verdict:         FAILED
CAAR mean:       -0.0000
CAAR p (canonical): 0.4363
BMP z:           +1.22  p=0.2234
event_hit_rate:  49.37%  p=0.0113
profit_factor:   0.993
clustering_hhi (normalized): 0.0015179888788612284


In [33]:
for d in ev_profile.diagnose():
    print(f'[{d.severity:<7s}] {d.code:<35s} {d.message}')

[veto   ] event.caar_bmp_sign_mismatch        CAAR and BMP have opposite signs — CAAR may be spuriously inflated by event-induced variance; use BMP as the trustworthy signal.
[warn   ] event.hit_rate_only                 Hit-rate is significant but CAAR and BMP are not — direction is correct on average, but economic magnitude is weak.
[warn   ] event.caar_trend_decay              CAAR trend is significantly negative — signal may be decaying.
[veto   ] event.oos_sign_flipped              OOS sample shows a sign flip vs IS — overfitting risk.


### 3.3 Level 2 — individual event metrics

In [34]:
from factorlib.metrics import (
    compute_caar, caar, bmp_test, event_hit_rate,
    corrado_rank_test, clustering_diagnostic,
)

ev_prepared = fl.preprocess(event_df_raw, config=ev_cfg)

caar_series = compute_caar(ev_prepared)   # per-event-date signed AR
print('caar:         ', caar(caar_series, forward_periods=5))
print('BMP:          ', bmp_test(ev_prepared, forward_periods=5))
print('event_hit_rate:', event_hit_rate(ev_prepared))
print('corrado rank: ', corrado_rank_test(ev_prepared))
print('clustering:   ', clustering_diagnostic(ev_prepared, cluster_window=3))

caar:          MetricOutput(caar=-0.0002, stat=-0.33)
BMP:           MetricOutput(bmp_sar=-0.0249, stat=-4.14, sig=***)
event_hit_rate: MetricOutput(event_hit_rate=0.4548, stat=-18.05, sig=***)
corrado rank:  MetricOutput(corrado_rank=-0.0136, stat=-9.41, sig=***)
clustering:    MetricOutput(clustering_hhi=0.0045)


### 3.4 Research session — `fl.factor()` for EventFactor

跟 CS 一樣，`fl.factor(df, name, config=cfg)` 回一個 `EventFactor` session —
所有 standalone event metric 都是 `f.<metric>()` 方法、統一回
`MetricOutput`。Path-based metric（`mfe_mae_summary` / `event_around_return` /
`multi_horizon_hit_rate`）在沒 `price` 欄位時 short-circuit；discrete
{±1} signal 時 `event_ic` 也 short-circuit。


In [ ]:
fe = fl.factor(event_df, 'GoldenCross', config=ev_cfg)
print('type:', type(fe).__name__)

# 全部 method 都回 MetricOutput（uniform contract）
for m in [fe.caar(), fe.bmp_test(), fe.event_hit_rate(), fe.profit_factor(),
          fe.event_skewness(), fe.signal_density(), fe.corrado_rank_test(),
          fe.caar_trend(), fe.oos_decay(), fe.mfe_mae_summary(),
          fe.event_around_return(), fe.multi_horizon_hit_rate()]:
    p = m.metadata.get('p_value')
    p_str = 'n/a' if p is None else f'{p:.4f}'
    reason = m.metadata.get('reason', '')
    val_str = f'{m.value:+.4f}' if isinstance(m.value, (int, float)) else str(m.value)[:40]
    print(f'  {m.name:<25s} value={val_str:>10s}  p={p_str:<8s} {reason}')

# event_ic 對 discrete signal short-circuits — reason 揭示原因
ic_out = fe.event_ic()
print(f'\n  event_ic              value={ic_out.value:+.4f}  reason={ic_out.metadata.get("reason")}')

# clustering_hhi 對 N>1 panel 有值；N=1 會 short-circuit
clu = fe.clustering_hhi()
print(f'  clustering_hhi        value={clu.value:+.4f}  reason={clu.metadata.get("reason", "")}')

# evaluate() reuse cache — 不會重算上面叫過的 metric
profile = fe.evaluate()
print(f'\nfe.evaluate() verdict: {profile.verdict()}, canonical_p: {profile.canonical_p:.4f}')
print(f'cache size: {len(fe.artifacts.metric_outputs)} metrics')


## 4. Macro panel（跨國/小截面配置因子）

訊號型態：連續值 + 小截面 `N < 30`。典型用法 = 跨國 CPI / 利差配置。
Canonical test: `fm_beta_p`（Fama-MacBeth Newey-West t-test）。

由於 TW 資料只有單一市場，這裡用 **TWSE 產業分類** 當 pseudo-country（N≈30）。

### 4.1 Build industry-aggregated panel

In [35]:
industry = (
    raw_demo.group_by(['date', 'twse_ind'])
    .agg(pl.col('price').mean())
    .rename({'twse_ind': 'asset_id'})
    .drop_nulls(['asset_id'])
    .sort(['asset_id', 'date'])
)
# factor = 產業相對 60 日均價偏離（簡單的 mean-reversion proxy）
industry = industry.with_columns(
    (pl.col('price') / pl.col('price').rolling_mean(60).over('asset_id') - 1).alias('factor')
).drop_nulls('factor')
print('industry N:', industry['asset_id'].n_unique())
industry.head(3)

industry N: 34


date,asset_id,price,factor
datetime[ms],str,f64,f64
2023-04-13 00:00:00,"""光電業""",57.617211,0.041827
2023-04-14 00:00:00,"""光電業""",57.151046,0.031346
2023-04-17 00:00:00,"""光電業""",58.176238,0.047598


### 4.2 evaluate + `MacroPanelConfig`

In [36]:
mp_cfg = fl.MacroPanelConfig(
    forward_periods=5,
    n_groups=3,               # 小截面 → 少分組
    demean_cross_section=False,
    min_cross_section=10,
)
industry_prep = fl.preprocess(industry, config=mp_cfg)
mp_profile = fl.evaluate(industry_prep, 'IndRelValue', config=mp_cfg)

print('verdict:              ', mp_profile.verdict())
print('fm_beta_mean (λ):     ', f'{mp_profile.fm_beta_mean:+.5f}')
print('fm_beta_tstat:        ', f'{mp_profile.fm_beta_tstat:+.2f}')
print('fm_beta_p (canonical):', f'{mp_profile.fm_beta_p:.4f}')
print('pooled_beta:          ', f'{mp_profile.pooled_beta:+.5f}  p={mp_profile.pooled_beta_p:.4f}')
print('beta_sign_consistency:', f'{mp_profile.beta_sign_consistency:.2%}')
print('q1_q5_spread:         ', f'{mp_profile.q1_q5_spread:+.4f}')
print('median cross-section N:', mp_profile.median_cross_section_n)


verdict:               FAILED
fm_beta_mean (λ):      +0.00009
fm_beta_tstat:         +0.82
fm_beta_p (canonical): 0.4121
pooled_beta:           -0.00002  p=0.7891
beta_sign_consistency: 54.30%
q1_q5_spread:          +0.0002
median cross-section N: 34


### 4.3 Diagnose + `fl.factor()` for MacroPanelFactor

`profile.diagnose()` 對 MP 也 work（same API）。`fl.factor()` 走 MacroPanelFactor
session：`fm_beta` / `pooled_beta` / `beta_sign_consistency` / `quantile_spread` /
`turnover` / `beta_trend` / `oos_decay` 都是 method。


In [ ]:
print('--- MP diagnose ---')
for d in mp_profile.diagnose():
    print(f'[{d.severity:<7s}] {d.code:<40s} {d.message}')

print('\n--- MacroPanelFactor session ---')
fmp = fl.factor(industry_prep, 'IndRelValue', config=mp_cfg)
for m in [fmp.fm_beta(), fmp.pooled_beta(), fmp.beta_sign_consistency(),
          fmp.quantile_spread(), fmp.turnover(), fmp.beta_trend(), fmp.oos_decay()]:
    p = m.metadata.get('p_value')
    p_str = 'n/a' if p is None else f'{p:.4f}'
    print(f'  {m.name:<25s} value={m.value:+.4f}  p={p_str}')


## 5. Macro common（全市場共用因子）

訊號型態：單一時序，每個資產共用同一個 factor 值。典型用法 = VIX / 黃金 / USD index。
Canonical test: `ts_beta_p`（per-asset 時序 OLS β 的截面 t-test）。

範例：市場截面波動率（所有資產日報酬的 cross-sectional std）→ 對每檔股票的 exposure 穩定嗎？

### 5.1 Build market vol + broadcast 到每個資產

In [37]:
# 市場波動率（每日：跨資產日報酬 std）
mkt_vol = (
    raw_demo.sort(['asset_id', 'date'])
    .with_columns(pl.col('price').pct_change().over('asset_id').alias('ret'))
    .group_by('date').agg(pl.col('ret').std().alias('factor'))
    .sort('date').drop_nulls('factor')
)
print('market vol head:')
print(mkt_vol.head(3))

# 為了控 runtime，sample 50 檔標的
sample_assets = raw_demo.select('asset_id').unique().head(50)['asset_id'].to_list()
common_df = (
    raw_demo.filter(pl.col('asset_id').is_in(sample_assets))
    .select(['date', 'asset_id', 'price'])
    .join(mkt_vol, on='date', how='inner')
)
print('common panel shape:', common_df.shape)

market vol head:
shape: (3, 2)
┌─────────────────────┬──────────┐
│ date                ┆ factor   │
│ ---                 ┆ ---      │
│ datetime[ms]        ┆ f64      │
╞═════════════════════╪══════════╡
│ 2023-01-04 00:00:00 ┆ 0.018249 │
│ 2023-01-05 00:00:00 ┆ 0.016922 │
│ 2023-01-06 00:00:00 ┆ 0.016153 │
└─────────────────────┴──────────┘
common panel shape: (17472, 4)


### 5.2 evaluate + `MacroCommonConfig`

In [38]:
mc_cfg = fl.MacroCommonConfig(
    forward_periods=5,
    ts_window=60,
    tradable=False,
)
common_prep = fl.preprocess(common_df, config=mc_cfg)
mc_profile = fl.evaluate(common_prep, 'MktVol', config=mc_cfg)

print('verdict:              ', mc_profile.verdict())
print('n_assets:             ', mc_profile.n_assets)
print('ts_beta_mean:         ', f'{mc_profile.ts_beta_mean:+.5f}')
print('ts_beta_tstat:        ', f'{mc_profile.ts_beta_tstat:+.2f}')
print('ts_beta_p (canonical):', f'{mc_profile.ts_beta_p:.4f}')
print('mean R²:              ', f'{mc_profile.mean_r_squared:.4f}')
print('beta_sign_consistency:', f'{mc_profile.ts_beta_sign_consistency:.2%}')


verdict:               PASS
n_assets:              50
ts_beta_mean:          -0.00073
ts_beta_tstat:         -3.87
ts_beta_p (canonical): 0.0003
mean R²:               0.0137
beta_sign_consistency: 76.00%


### 5.3 Diagnose + `fl.factor()` for MacroCommonFactor

`ts_beta` / `mean_r_squared` / `ts_beta_sign_consistency` / `beta_trend` /
`oos_decay` 統一走 method。N=1 degenerate case 由 primitive 處理，Factor 不
重複邏輯。


In [ ]:
print('--- MC diagnose ---')
for d in mc_profile.diagnose():
    print(f'[{d.severity:<7s}] {d.code:<40s} {d.message}')

print('\n--- MacroCommonFactor session ---')
fmc = fl.factor(common_prep, 'MktVol', config=mc_cfg)
for m in [fmc.ts_beta(), fmc.mean_r_squared(), fmc.ts_beta_sign_consistency(),
          fmc.beta_trend(), fmc.oos_decay()]:
    p = m.metadata.get('p_value')
    p_str = 'n/a' if p is None else f'{p:.4f}'
    print(f'  {m.name:<25s} value={m.value:+.4f}  p={p_str}')


## 6. 切換因子類別的 cheat sheet

只要換 `factor_type=` 字串（或對應的 `XxxConfig`），同一組 `fl.evaluate` / `fl.evaluate_batch` / `ProfileSet` API 都能用——差別在回傳的 Profile dataclass 欄位不同、canonical p 對應的統計量不同。

In [39]:
cheat = pl.DataFrame({
    'factor_type': ['cross_sectional', 'event_signal', 'macro_panel', 'macro_common'],
    'signal_shape': ['每期每資產連續值', '離散 {-1,0,+1}', '連續值，小截面 N<30', '單一時序、全資產共用'],
    'Config':       ['CrossSectionalConfig', 'EventConfig', 'MacroPanelConfig', 'MacroCommonConfig'],
    'Profile':      ['CrossSectionalProfile', 'EventProfile', 'MacroPanelProfile', 'MacroCommonProfile'],
    'canonical_p':  ['ic_p', 'caar_p', 'fm_beta_p', 'ts_beta_p'],
    'entry_api':    ['fl.factor / fl.evaluate', 'fl.factor / fl.evaluate',
                     'fl.factor / fl.evaluate', 'fl.factor / fl.evaluate'],
    'core_question': ['排序能預測截面報酬差異嗎？', '事件後報酬有異常嗎？',
                       '宏觀指標能預測跨資產配置嗎？', '資產對共同因子的 exposure 穩定嗎？'],
})
cheat


factor_type,signal_shape,Config,Profile,canonical_p,core_question
str,str,str,str,str,str
"""cross_sectional""","""每期每資產連續值""","""CrossSectionalConfig""","""CrossSectionalProfile""","""ic_p""","""排序能預測截面報酬差異嗎？"""
"""event_signal""","""離散 {-1,0,+1}""","""EventConfig""","""EventProfile""","""caar_p""","""事件後報酬有異常嗎？"""
"""macro_panel""","""連續值，小截面 N<30""","""MacroPanelConfig""","""MacroPanelProfile""","""fm_beta_p""","""宏觀指標能預測跨資產配置嗎？"""
"""macro_common""","""單一時序、全資產共用""","""MacroCommonConfig""","""MacroCommonProfile""","""ts_beta_p""","""資產對共同因子的 exposure 穩定嗎？"""


---

## 附錄 A：全 factor_type 收到的 Profile 一覽

把本 notebook 跑出來的四個 profile 並排，展示不同 factor_type 的 verdict + canonical_p 是同一套介面。

In [40]:
summary = pl.DataFrame([
    {
        'factor_type': 'cross_sectional',
        'factor_name': profile.factor_name,
        'verdict': profile.verdict(),
        'canonical_p': profile.canonical_p,
        'n_periods': profile.n_periods,
    },
    {
        'factor_type': 'event_signal',
        'factor_name': ev_profile.factor_name,
        'verdict': ev_profile.verdict(),
        'canonical_p': ev_profile.canonical_p,
        'n_periods': ev_profile.n_periods,
    },
    {
        'factor_type': 'macro_panel',
        'factor_name': mp_profile.factor_name,
        'verdict': mp_profile.verdict(),
        'canonical_p': mp_profile.canonical_p,
        'n_periods': mp_profile.n_periods,
    },
    {
        'factor_type': 'macro_common',
        'factor_name': mc_profile.factor_name,
        'verdict': mc_profile.verdict(),
        'canonical_p': mc_profile.canonical_p,
        'n_periods': mc_profile.n_periods,
    },
])
summary

factor_type,factor_name,verdict,canonical_p,n_periods
str,str,str,f64,i64
"""cross_sectional""","""Mom_20D""","""FAILED""",0.725113,330
"""event_signal""","""GoldenCross""","""FAILED""",0.436297,331
"""macro_panel""","""IndRelValue""","""FAILED""",0.412085,291
"""macro_common""","""MktVol""","""PASS""",0.000326,349


---

## 附錄 B：`describe_profile_values` × 4 factor types

同一個 `fl.describe_profile_values(profile, artifacts)` 吃四種 Profile 都 work——
scalar 欄位自動根據該 factor_type 的 canonical metric list 渲染；opt-in detail
section（regime / multi_horizon / spanning）有就印、沒有就跳過。

這裡重跑 4 個 factor type 的 `evaluate(..., return_artifacts=True)` 拿到
artifacts，展示同一 API 的跨型別一致性。

In [41]:
cs_p,  cs_a  = fl.evaluate(mom20,         'Mom_20D',      factor_type='cross_sectional', return_artifacts=True)
ev_p,  ev_a  = fl.evaluate(event_df,      'GoldenCross',  config=ev_cfg,                 return_artifacts=True)
mp_p,  mp_a  = fl.evaluate(industry_prep, 'IndRelValue',  config=mp_cfg,                 return_artifacts=True)
mc_p,  mc_a  = fl.evaluate(common_prep,   'MktVol',       config=mc_cfg,                 return_artifacts=True)

for p, a in [(cs_p, cs_a), (ev_p, ev_a), (mp_p, mp_a), (mc_p, mc_a)]:
    fl.describe_profile_values(p, a, include_detail=False)



  Mom_20D — CrossSectionalProfile  (n_periods=330, verdict=FAILED)
  ------------------------------------------------------------
  Metrics:
    ic                 value=-0.0098    stat=-0.3532  sig=     p=0.7251
    ic_ir              value=-0.1025    stat=-        sig=     p=-
    hit_rate           value=0.5000     stat=0.0000   sig=     p=1.0000
    ic_trend           value=-0.0001    stat=-1.3013  sig=     p=0.1941
    monotonicity       value=0.4645     stat=2.7904   sig=***  p=0.0069
    q1_q5_spread       value=0.0009     stat=2.0571   sig=**   p=0.0437
    turnover           value=0.0734     stat=-        sig=     p=-
    breakeven_cost     value=61.9188    stat=-        sig=     p=-
    net_spread         value=0.0005     stat=-        sig=     p=-
    q1_concentration   value=177.3400   stat=311.7139 sig=     p=1.0000


  GoldenCross — EventProfile  (n_periods=331, verdict=FAILED)
  ------------------------------------------------------------
  Metrics:
    caar            

---

## 7. Library helpers（非 per-type 的 cross-cutting API）

本節集中展示 factorlib 的「library-level」工具——這些不屬於某個 factor_type
的 Profile / Factor，但在 research workflow 常用：

- `fl.split_by_group` — 按條件把 panel 切 sub-universe + N-aware config
- `fl.validate_factor_data` — pandera schema check
- `fl.MARKET_DEFAULTS` — 每個市場的 estimated_cost_bps 預設
- `fl.register_rule` / `fl.clear_custom_rules` — 使用者自訂 diagnose rule
- `evaluate_batch(compact=True)` — 大批次省記憶體
- `evaluate_batch(on_result=, on_error=, stop_on_error=)` — streaming callback
- `redundancy_matrix(method='factor_rank')` — 替代 `value_series` 的冗餘算法
- `multiple_testing_correct(p_source=...)` — 改 BHY 對照的 p-value 欄位


### 7.1 `split_by_group` — sub-universe 切分 + N-aware n_groups

把一個大 panel 依 filter 條件切成多個 sub-universe（e.g. 大型股 / 中小型股），
自動調整 `n_groups` 讓小 N 不至於用 decile（通常會退到 quintile 或 tercile）。


In [ ]:
from factorlib import split_by_group

# 用 mom20 的 prepared panel 當 base；切成前 1/3 vs 後 2/3 by factor
split_cfg = fl.CrossSectionalConfig(forward_periods=5, n_groups=10)
q_hi = mom20['factor'].quantile(2/3)
groups = split_by_group(
    mom20,
    definitions={
        'high_mom':  pl.col('factor') >= q_hi,
        'low_mom':   pl.col('factor') <  q_hi,
    },
    base_config=split_cfg,
)
for name, (sub_df, sub_cfg) in groups.items():
    print(f'  {name:<10s} rows={sub_df.height:<5d} n_groups={sub_cfg.n_groups}  '
          f'(base n_groups={split_cfg.n_groups})')


### 7.2 `validate_factor_data` — pandera schema check

Per-factor-type schema check — 用在 pipeline 入口、catch 壞資料。


In [ ]:
# raw_demo 是 canonical CS schema，檢查通過
fl.validate_factor_data(raw_demo, factor_type='cross_sectional')
print('raw_demo: CS schema OK')

# 缺欄位會 raise
try:
    fl.validate_factor_data(raw_demo.drop('asset_id'), factor_type='cross_sectional')
except Exception as e:
    print(f'drop(asset_id) → {type(e).__name__}: {str(e)[:80]}...')


### 7.3 `MARKET_DEFAULTS` — 市場預設 cost

不同市場 cost 差距大（TW 30bps、US 5bps）。用 `MARKET_DEFAULTS` 避免 magic
number 散落 notebook。


In [ ]:
print('MARKET_DEFAULTS:', fl.MARKET_DEFAULTS)
cfg_tw = fl.CrossSectionalConfig(**fl.MARKET_DEFAULTS['tw'])
cfg_us = fl.CrossSectionalConfig(**fl.MARKET_DEFAULTS['us'])
print(f'TW cost: {cfg_tw.estimated_cost_bps} bps')
print(f'US cost: {cfg_us.estimated_cost_bps} bps')


### 7.4 `register_rule` — 自訂 diagnose rule

使用者自定義的統計規則可以用 `register_rule` 掛進 diagnose pipeline，
`profile.diagnose()` 會自動呼叫。`clear_custom_rules(...)` 清掉（避免汙染其他
analysis）。


In [ ]:
from factorlib import Rule, register_rule, clear_custom_rules
from factorlib.evaluation.profiles import CrossSectionalProfile
from factorlib._types import Diagnostic

def _low_ic_ir_rule(profile: CrossSectionalProfile, artifacts) -> Diagnostic | None:
    if abs(profile.ic_ir) < 0.15:
        return Diagnostic(
            severity='caution',
            code='cs.user_low_ic_ir',
            message=f'IC_IR={profile.ic_ir:.3f} below user threshold 0.15',
        )
    return None

register_rule(CrossSectionalProfile, Rule(
    fn=_low_ic_ir_rule, severity='caution', code='cs.user_low_ic_ir',
))

# Re-run diagnose — 自訂 rule 會跟 built-in 一起出現
for d in profile.diagnose():
    marker = ' ★' if d.code == 'cs.user_low_ic_ir' else ''
    print(f'[{d.severity:<7s}] {d.code:<35s}{marker}')

# 收尾：清掉 custom rule，不影響後續 analysis
clear_custom_rules(CrossSectionalProfile)


### 7.5 `compact=True` — 大批次省記憶體

`evaluate_batch(..., compact=True, keep_artifacts=True)` 把每個 `Artifacts` 的
`prepared` DataFrame 換成 sentinel（丟掉佔 MB 的 raw panel、保留 KB 級的
`intermediates` + `metric_outputs`）。對 >50 factor 的批次幾乎必開。

存取 `artifacts.prepared` 會 raise 並告知是 `compact=True` 造成——err message
直接指向修正路徑。


In [ ]:
# 先跑一版不 compact 的當 baseline
ps_normal, arts_normal = fl.evaluate_batch(
    factors_map, factor_type='cross_sectional', keep_artifacts=True,
)

# 再跑一版 compact=True
ps_compact, arts_compact = fl.evaluate_batch(
    factors_map, factor_type='cross_sectional',
    keep_artifacts=True, compact=True,
)

# Normal path 可以摸 prepared；compact path 會 raise
sample_name = next(iter(arts_normal))
print(f'normal  arts_normal["{sample_name}"].prepared.height = '
      f'{arts_normal[sample_name].prepared.height} rows')

try:
    _ = arts_compact[sample_name].prepared.columns
except RuntimeError as e:
    print(f'compact RuntimeError (truncated): {str(e)[:120]}...')

# intermediates / metric_outputs 在 compact 下仍可用——所以
# redundancy_matrix(method='value_series') 相容 compact
redund_c = fl.redundancy_matrix(ps_compact, method='value_series', artifacts=arts_compact)
print(f'\nredundancy under compact=True: shape={redund_c.shape}')


### 7.6 `evaluate_batch` callbacks — `on_result` / `on_error` / `stop_on_error`

`on_result(name, profile)` 每個 factor 算完觸發（log / MLflow / 進度條）。
Return `False` 可以中止整個 batch（其他 truthy / None 值繼續）。
`stop_on_error=False`（default）搭配 `on_error` 拿到失敗 factor 的 exception，
loop 不中斷。


In [ ]:
# 裝個壞 factor 進 map 測 on_error
bad_df = mom20.drop('forward_return')   # 缺 forward_return → 會炸 strict gate
factors_with_bad = {**factors_map, 'broken': bad_df}

progress_log = []
errors_log = []

def _on_result(name, prof):
    progress_log.append((name, prof.verdict()))
    return None  # 繼續；return False 會停

def _on_error(name, exc):
    errors_log.append((name, type(exc).__name__, str(exc)[:60]))

ps_cb = fl.evaluate_batch(
    factors_with_bad,
    factor_type='cross_sectional',
    stop_on_error=False,
    on_result=_on_result,
    on_error=_on_error,
)
print('progress:', progress_log)
print('errors:  ', errors_log)
print(f'n factors in ProfileSet: {len(ps_cb)} (壞 factor 被 skip)')


### 7.7 `redundancy_matrix(method='factor_rank')` — 替代算法

`method='value_series'`（demo 2.6 用的）對 `intermediates[value_series]` 跑
pairwise `|ρ|`，單位：每個 factor 的 IC / CAAR series。
`method='factor_rank'` 改對 `prepared` 裡每日的 factor rank 算——同一面板上
的真實因子相關性，跟 value_series 互補。factor_rank 需要 `prepared`，
**不能** 搭配 `compact=True`。


In [ ]:
# 用 normal（non-compact）artifacts 跑 factor_rank
redund_fr = fl.redundancy_matrix(ps_normal, method='factor_rank', artifacts=arts_normal)
print('factor_rank:')
print(redund_fr)

print('\nvalue_series (demo §2.6 已跑過; 同一組 factor):')
print(redund)


### 7.8 `multiple_testing_correct(p_source=...)` — 選擇 BHY 對照欄位

`canonical_p`（default）對每個 factor_type 對應 one canonical test。想對
另一 p-value 欄位做 BHY（e.g. CS 的 `spread_p` 代表 Q1-Q5 spread alpha），
可傳 `p_source='spread_p'`。白名單在 `Profile.P_VALUE_FIELDS`；不在白名單
會 raise。


In [ ]:
# CS 的 P_VALUE_FIELDS
print('CS P_VALUE_FIELDS:', sorted(fl.CrossSectionalProfile.P_VALUE_FIELDS))

# 對 spread_p 做 BHY（跟對 ic_p 不同 question）
adj_spread = ps.multiple_testing_correct(p_source='spread_p', fdr=0.10)
print('\nBHY on spread_p:')
print(adj_spread.to_polars().select('factor_name', 'spread_p', 'p_adjusted', 'bhy_significant'))

# 不在白名單的欄位會 raise
try:
    ps.multiple_testing_correct(p_source='ic_mean', fdr=0.10)
except ValueError as e:
    print(f'\nbad p_source raise: {str(e)[:100]}...')
